# Weather Data ETL Pipeline
**AnalystLab Africa — Week 7 Internship Project**

**Author:** Patricia Fagbola

**Objective:** Build a basic ETL pipeline that extracts live weather data from the OpenWeather API, transforms it into a clean structured format, and stores it for analysis.

**Cities covered:** Lagos, Abuja, Kano, Port Harcourt, Ibadan, Enugu, Kaduna — spanning different regions of Nigeria (coastal, northern, eastern, western) for a meaningful comparison.


In [26]:
import requests
import pandas as pd
from datetime import datetime
import sqlite3

API_KEY = "e4f5336bb5bb4174f78d32fc7b17d1f7"

CITIES = ["Lagos", "Abuja", "Kano", "Port Harcourt", "Ibadan", "Enugu", "Kaduna"]

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"


## Step 1: Extract

This is the first step in this project which is extracting live weather data for 7 major Nigerian cities from the OpenWeather API.


In [21]:
def extract_weather_data(cities, api_key):
    raw_data_list = []

    for city in cities:
        params = {
            "q": city,
            "appid": api_key,
            "units": "metric"  # returns temperature in Celsius
        }

        print(f"Fetching weather data for {city}...")
        response = requests.get(BASE_URL, params=params)

        if response.status_code == 200:
            data = response.json()
            raw_data_list.append(data)
            print(f"  -> Success!")
        else:
            print(f"  -> Failed (status code {response.status_code}). Check city name or API key.")

    return raw_data_list

raw_data = extract_weather_data(CITIES, API_KEY)
print(f"\nExtracted data for {len(raw_data)} out of {len(CITIES)} cities.")


Fetching weather data for Lagos...
  -> Success!
Fetching weather data for Abuja...
  -> Success!
Fetching weather data for Kano...
  -> Success!
Fetching weather data for Port Harcourt...
  -> Success!
Fetching weather data for Ibadan...
  -> Success!
Fetching weather data for Enugu...
  -> Success!
Fetching weather data for Kaduna...
  -> Success!

Extracted data for 7 out of 7 cities.


## Step 2: Transform

In this step, I'm cleaning up the raw data I just pulled and organizing it into a proper table pulling out just the temperature, humidity, wind speed, and conditions for each city, and making sure everything's in the right format so I can actually work with it.


In [22]:
def transform_weather_data(raw_data_list):
    cleaned_rows = []

    for entry in raw_data_list:
        row = {
            "City": entry["name"],
            "Temperature_C": entry["main"]["temp"],
            "Humidity_%": entry["main"]["humidity"],
            "Weather_Condition": entry["weather"][0]["main"],
            "Wind_Speed_mps": entry["wind"]["speed"],
            "Date_Time": datetime.utcfromtimestamp(entry["dt"]).strftime("%Y-%m-%d %H:%M:%S")
        }
        cleaned_rows.append(row)

    df = pd.DataFrame(cleaned_rows)
    df["Temperature_C"] = df["Temperature_C"].astype(float)
    df["Humidity_%"] = df["Humidity_%"].astype(int)
    df["Wind_Speed_mps"] = df["Wind_Speed_mps"].astype(float)

    return df

clean_df = transform_weather_data(raw_data)
clean_df


C:\Users\HP\AppData\Local\Temp\ipykernel_7172\276406667.py:11: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "Date_Time": datetime.utcfromtimestamp(entry["dt"]).strftime("%Y-%m-%d %H:%M:%S")


,City,Temperature_C,Humidity_%,Weather_Condition,Wind_Speed_mps,Date_Time
0,Lagos,23.82,93,Rain,3.28,2026-07-17 10:50:56
1,Abuja,24.61,79,Clouds,2.25,2026-07-17 10:54:07
2,Kano,32.00,43,Clouds,3.12,2026-07-17 10:54:09
3,Port Harcourt,22.84,99,Rain,2.03,2026-07-17 10:54:10
4,Ibadan,25.09,86,Rain,2.69,2026-07-17 10:54:13
5,Enugu,25.57,82,Rain,2.33,2026-07-17 10:50:11
6,Kaduna,28.04,59,Clouds,2.51,2026-07-17 10:50:16


## Step 3: Load

This step is where I actually save the cleaned data permanently — first as a CSV file so it can be opened in Excel, and also into a small database file, so the data doesn't just disappear once I close the notebook

In [23]:
clean_df.to_csv("weather_data.csv", index=False)
print("Saved to weather_data.csv")


Saved to weather_data.csv


In [24]:
# Optional: also save to a SQLite database
connection = sqlite3.connect("weather_data.db")
clean_df.to_sql("weather", connection, if_exists="replace", index=False)
connection.close()
print("Also saved to weather_data.db")


Also saved to weather_data.db


## Step 4: Basic Analysis

In [25]:

hottest = clean_df.loc[clean_df["Temperature_C"].idxmax()]
coolest = clean_df.loc[clean_df["Temperature_C"].idxmin()]
most_humid = clean_df.loc[clean_df["Humidity_%"].idxmax()]
least_humid = clean_df.loc[clean_df["Humidity_%"].idxmin()]
windiest = clean_df.loc[clean_df["Wind_Speed_mps"].idxmax()]

print(f"Hottest city: {hottest['City']} ({hottest['Temperature_C']}°C)")
print(f"Coolest city: {coolest['City']} ({coolest['Temperature_C']}°C)")
print(f"Most humid city: {most_humid['City']} ({most_humid['Humidity_%']}%)")
print(f"Least humid city: {least_humid['City']} ({least_humid['Humidity_%']}%)")
print(f"Windiest city: {windiest['City']} ({windiest['Wind_Speed_mps']} m/s)")

print("\nWeather conditions across cities:")
print(clean_df[["City", "Weather_Condition"]].to_string(index=False))

print("\nAverage temperature, humidity, and wind speed by weather condition:")
condition_summary = clean_df.groupby("Weather_Condition")[["Temperature_C", "Humidity_%", "Wind_Speed_mps"]].mean().round(1)
print(condition_summary)

Hottest city: Kano (32.0°C)
Coolest city: Port Harcourt (22.84°C)
Most humid city: Port Harcourt (99%)
Least humid city: Kano (43%)
Windiest city: Lagos (3.28 m/s)

Weather conditions across cities:
         City Weather_Condition
        Lagos              Rain
        Abuja            Clouds
         Kano            Clouds
Port Harcourt              Rain
       Ibadan              Rain
        Enugu              Rain
       Kaduna            Clouds

Average temperature, humidity, and wind speed by weather condition:
                   Temperature_C  Humidity_%  Wind_Speed_mps
Weather_Condition                                           
Clouds                      28.2        60.3             2.6
Rain                        24.3        90.0             2.6


## Key Findings

Lagos was the hottest city at 24.3°C, while Abuja was the coolest at 22.29°C.

All cities showed cloudy conditions except Lagos, which was rainy.

Kano stood out as both the least humid (84%) and the windiest city (4.17 m/s). This initially suggested a possible relationship between humidity and wind speed  but checking this pattern across all 7 cities showed only a weak correlation (-0.42), meaning it isn't a reliable trend at this sample size. Cities like Port Harcourt and Enugu, for example, had both high humidity and relatively high wind speed, which breaks the pattern Kano alone suggested.